# Train and evaluate ReefStudent (Stories 7.2 and 7.3)

Kaggle T4, internet on. Attach the private dataset `aditya32211/reefcase-kaggle`, which holds `src/reefcase`, `ml/` and the SAM 3 outputs of `ml/modal/pseudo_label_run.py` (`trainval/` pseudo-labels, `sam3_test/` test-split predictions). Coralscapes itself comes from Hugging Face (Apache-2.0); the test split is used only for evaluation.

Two students, same architecture and recipe:
- **distilled**: trained on SAM 3 pseudo-labels only. If the dataset already holds `distilled/reefstudent.pt` from an earlier run, it is reused, not retrained.
- **supervised**: trained on the Coralscapes human labels, mapped to six classes with `ml/coralscapes_map.json` (unlabelled pixels are ignored).

Outputs in `/kaggle/working`: `distilled/` and `supervised/` (`reefstudent.pt`, `curve.json`, `metrics.json`) and `eval.json`, which scores both students and SAM 3 against the human test labels.

In [ ]:
import glob
import io
import json
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

W = Path("/kaggle/working")
DATA = Path("/tmp/coralscapes")  # frames stay out of /kaggle/working, so they are not saved
for archive in glob.glob("/kaggle/input/**/*.zip", recursive=True):
    zipfile.ZipFile(archive).extractall("/tmp/unzipped")


def find(pattern, required=True):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    hits += sorted(glob.glob(f"/tmp/unzipped/**/{pattern}", recursive=True))
    assert hits or not required, f"not in the attached dataset: {pattern}"
    return Path(hits[0]) if hits else None


SRC = find("reefcase/__init__.py").parents[1]
TRAINVAL = find("trainval/manifest.jsonl").parent
SAM3 = find("sam3_test/timing.json").parent
DISTILLED = find("distilled/reefstudent.pt", required=False)
shutil.copytree(find("kaggle/train_student.py").parents[1], W / "ml", dirs_exist_ok=True)
os.environ["PYTHONPATH"] = str(SRC)
sys.path[:0] = [str(SRC), str(W / "ml/kaggle")]
pip = [sys.executable, "-m", "pip", "install", "-q"]
subprocess.run([*pip, "segmentation-models-pytorch==0.6.0", "pydantic-settings"], check=True)

import torch  # noqa: E402

print(sys.version, torch.__version__, torch.cuda.get_device_name(0))
print("reusing distilled weights:", DISTILLED)

In [ ]:
import numpy as np
import pyarrow.parquet as pq
from eval_student import human_lut
from huggingface_hub import HfApi, hf_hub_download
from PIL import Image

REPO = "EPFL-ECEO/coralscapes"
FILES = sorted(HfApi().list_repo_files(REPO, repo_type="dataset"))
LUT = human_lut(json.loads((W / "ml/coralscapes_map.json").read_text(encoding="utf-8")))


def dump(split, frames, labels, mapped):
    """Write frames and labels under the image's name; ``mapped`` applies the six-class LUT."""
    frames.mkdir(parents=True, exist_ok=True)
    labels.mkdir(parents=True, exist_ok=True)
    names = []
    for file in (f for f in FILES if f.startswith(f"data/{split}-")):
        local = hf_hub_download(REPO, file, repo_type="dataset", local_dir="/tmp/hf")
        table = pq.read_table(local)
        pairs = zip(
            table.column("image").to_pylist(), table.column("label").to_pylist(), strict=True
        )
        for image, label in pairs:
            name = image["path"]
            (frames / name).write_bytes(image["bytes"])
            if mapped:
                ids = np.asarray(Image.open(io.BytesIO(label["bytes"])))
                assert ids.max() <= 39, f"{name}: not a Coralscapes label"
                Image.fromarray(LUT[ids], mode="L").save(labels / name)
            else:
                (labels / name).write_bytes(label["bytes"])
            names.append(name)
        os.remove(local)
    return names


HUMAN = DATA / "human"
rows = [
    {"frame": name, "mask": f"labels/{name}", "source": "coralscapes-human"}
    for split in ("train", "validation")
    for name in dump(split, DATA / "frames", HUMAN / "labels", mapped=True)
]
(HUMAN / "manifest.jsonl").write_text("".join(json.dumps(row) + "\n" for row in rows))
print("train+validation", len(rows))
print("test", len(dump("test", DATA / "test/frames", DATA / "test/labels", mapped=False)))

In [ ]:
def train(manifest, out):
    script = W / "ml/kaggle/train_student.py"
    args = ["--manifest", manifest, "--frames", DATA / "frames", "--out", out, "--epochs", "30"]
    subprocess.run([sys.executable, script, *args], check=True)


if DISTILLED:
    shutil.copytree(DISTILLED.parent, W / "distilled", dirs_exist_ok=True)
else:
    train(TRAINVAL / "manifest.jsonl", W / "distilled")
train(HUMAN / "manifest.jsonl", W / "supervised")

In [ ]:
students = [f"{name}={W / name / 'reefstudent.pt'}" for name in ("distilled", "supervised")]
subprocess.run(
    [
        sys.executable,
        W / "ml/kaggle/eval_student.py",
        *[arg for spec in students for arg in ("--weights", spec)],
        "--test",
        DATA / "test",
        "--sam3",
        SAM3,
        "--out",
        W / "eval.json",
    ],
    check=True,
)